### **Setup: Imports und Initialisierung**

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
from quinex_application.quinex_client import QuinexClient
from quinex_application.metadata_visualisation import VisualisationMetadata
from quinex_application.mongos_store import MongoStore

In [2]:
quinex = QuinexClient()
visualisation = VisualisationMetadata("../data/paper/paper_dataset.parquet")
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_wos_abstract")

### **Create Bulk Analysis**

In [3]:
name = "quinex_application_query4"
response = quinex.intialize_analysis(name)
print(response)

{'detail': 'Analysis with name quinex_application_query4 already exists. Please choose another name.'}


### **Import and Sort Paper by Year**

In [6]:
visualisation.load_df(query='Query: AB=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))', date="10.06.2025")
df_doi = visualisation.df
len(df_doi)

17173

In [5]:
visualisation.load_df(query='Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )', date="14.05.2025")
df_doi = visualisation.df
len(df_doi)

25384

In [ ]:
 # Sort by year
dfs_by_year = {
    year: group_df
    for year, group_df in df_doi.groupby("Year")
}
df_2010 = dfs_by_year[2010]
print(len(df_2010))

## **Add Papers via DOI**

In [ ]:
mongodb.insert_paper(name, df_2025)

#### **Finish remaining Papers**

In [ ]:
# Hole die DOIs (_id) aus beiden Collections als Sets
papers_ids = {doc["_id"] for doc in mongodb.get_collection("papers_wos")}
failed_papers_ids = {doc["_id"] for doc in mongodb.get_collection("failed_papers")}
#papers_2025_abstract_ids = {doc["_id"] for doc in mongodb.get_collection("papers_2025_abstract")}

# Vereine beide Sets (alle DOIs, die ausgeschlossen werden sollen)
excluded_ids = papers_ids.union(failed_papers_ids)

# Filtere das DataFrame: nur Zeilen mit DOIs, die NICHT in excluded_ids enthalten sind
filtered_df = df_doi[~df_doi["DOI"].isin(excluded_ids)]
failed_df = df_doi[df_doi["DOI"].isin(failed_papers_ids)]
print(len(filtered_df))
print(len(failed_df))

In [ ]:
mongodb.insert_paper(name, failed_df, take_abstract=True)

## **Add Papers via Scopus CSV**

In [8]:
df = pd.read_excel("../data/paper/wos_query4_abstracts/wos.xls") 
mongodb.insert_scopus_csv(name, df, force_overwrite=True)

✅ Inserting papers finished: {'detail': 'Added 517 of 517 papers to the analysis "quinex_application_query4"', 'paper_ids': ['W_sha256_d30ce93db7be68b07a1459a548ec79d69461348f2516e31113912061609738a5', 'W_sha256_b2abb2174616eb3cc59a2c1ffd67b8034ebb7c847adf9022065d9af5423b42f0', 'W_sha256_7e22879f2fef22c88969de480eb0a825a72419b39357bb0d43ce33eca4b7e7e5', 'W_sha256_41a5349a9e5507ed2e8b25c2615637fb46b57b468e6403d3278b1d4fe81a7504', 'W_sha256_07dd9490086707b0a54066b281c8edaa096921e88ca31f0f65486e492621bc74', 'W_sha256_642861157ba856a433caec50de7e0a568dd11605d955ed0e7919d431e45e9c6a', 'W_sha256_930dd4bb0c4c520635c2353693d91dddf6616af19172624255b6a99a1aeff36d', 'W_sha256_da7b6733682f6f374ecec8bfb499181ba8fe5cc658cffbfaeaa2d96718734eed', 'W_sha256_27fe5c129dd0c9b60cae9d1fbe706b0873902c8abd054acd9e7671b0d6536904', 'W_sha256_33f7e8399613c136994e2c015150e8e425f9df45c221a03002d70dad19ff2b2c', 'W_sha256_23169ad860eae43eb5ecbc192b006287076784929b870628d67086b971f3ec2a', 'W_sha256_e50f35889ff0146392

In [13]:
collection = mongodb.get_collection("failed_papers")
len(collection)

92

### **Successfully Added Papers**

In [16]:
dois_df = mongodb.get_papers_from_collection(df_doi, "papers_wos_abstract", False)
print(len(dois_df))

517


## **Perform Bulk Analysis**

In [17]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.reset_quinex_result(dois)
mongodb.process_paper(name, dois, number_of_gpus=8)

🔄 Reset: 274 of 517 requested DOIs.
✅ Analysis finished: {'detail': 'Analysis quinex_application_query4 was successfully processed.', 'processed_papers': ['W4406848651', 'W4400787440', 'W4398238984', 'W4398253035', 'W4323025856', 'W4362714345', 'W4302362391', 'W3168157123', 'W3181201154', 'W3034194289', 'W3034298551', 'W3048508027', 'W3045327901', 'W3080455641', 'W3013233328', 'W2996809290', 'W3115906263', 'W3080528870', 'W3046623886', 'W3082808731', 'W3087905503', 'W3090282993', 'W3016723514', 'W3019382326', 'W3080632812', 'W2998197160', 'W3024532996', 'W2964797642', 'W2912270904', 'W2800410101', 'W2079157570', 'W2065428201', 'W2030155416', 'W2283476039', 'W2411267848', 'W2770799738', 'W2791174276', 'W2794587103', 'W2959579004', 'W3034822143', 'W2981367174', 'W2995883691', 'W3095246848', 'W2993090621', 'W3020787492', 'W3093504391', 'W3203760898', 'W3158920843', 'W3166960374', 'W3186267768', 'W3214024635', 'W3195075485', 'W3207677299', 'W4205697628', 'W4229331787', 'W4210932418', 'W429

## **Add Bulk Analysis to MongoDB**

In [18]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.add_analysis(name, dois)


📊 Successfully processed: 517 of 517 DOIs


## **Update Paper Parquet**

In [67]:
df = mongodb.update_paper_parquet("papers_wos_abstract", "../data/paper/paper_dataset.parquet", "15.10.2025", "17.04.2025", abstract=True)

In Collection papers_wos_abstract -> Quinex True: 335, Quinex False: 182


In [68]:
print("-------------------QUINEX Performed-------------------------------")
print(len(df[(df["Quinex"] == True)]))
print("-------------------QUINEX Tried-----------------------------------")
print(len(df[(df["Download_Status"] == "Succeeded")]) +  len(df[(df["Download_Status"] == "Abstract Succeeded")]) + len(df[(df["Download_Status"] == "Failed")]))
print("-------------------Overall Papers-------------------------------")
print(len(df[(df['Query: AB=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_10.06.2025'] == True) | (df['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_14.05.2025'] == True)]))

-------------------QUINEX Performed-------------------------------
21134
-------------------QUINEX Tried-----------------------------------
25585
-------------------Overall Papers-------------------------------
26248


In [69]:
print(len(df[(df["Download_Status"] == "Abstract Succeeded")]))
print(len(df[(df["Download_Status"] == "Succeeded")]))
print(len(df[(df["Download_Status"] == "Failed")]))

11035
14494
56
